In [26]:
%%writefile main.py

from fastapi import FastAPI
from pydantic import BaseModel, Field
import pandas as pd
import joblib

# Load the Ride Booking ML model
model = joblib.load("catboost_ride_booking_model.joblib")

# Create FastAPI application
app = FastAPI(
    title="Ride Booking Classification API",
    description="Predict ride booking status using a trained CatBoost model",
    version="1.0.0"
)

# Define input features
class RideBookingInput(BaseModel):
    vehicle_type: str
    pickup_location: str
    drop_location: str
    booking_hour: int = Field(ge=0, le=23)
    day_of_week: int = Field(ge=0, le=6)
    is_weekend: int = Field(ge=0, le=1)
    month_name: str

# Home endpoint
@app.get("/")
def home():
    return {
        "message": "Ride Booking Classification API is running!"
    }

# Health endpoint
@app.get("/health")
def health():
    return {
        "status": "healthy",
        "model": "CatBoost V2"
    }

# Prediction endpoint
@app.post("/predict")
def predict_booking(data: RideBookingInput):
    input_data = pd.DataFrame([data.model_dump()])

    prediction = model.predict(input_data)
    predicted_status = str(prediction.reshape(-1)[0])

    probabilities = model.predict_proba(input_data)[0]
    confidence = float(max(probabilities))

    return {
        "predicted_booking_status": predicted_status,
        "prediction_probability": round(confidence, 4)
    }

Overwriting main.py


In [27]:
process.terminate()
process.wait()

print("Old API server stopped!")

Old API server stopped!


In [35]:
import requests

for port in [8000, 8001]:
    try:
        response = requests.get(
            f"http://127.0.0.1:{port}/",
            timeout=3
        )
        print(f"\nPort: {port}")
        print("Status:", response.status_code)
        print("Response:", response.text[:300])
    except Exception as e:
        print(f"Port {port}: {e}")


Port: 8000
Status: 200
Response: {"message":"Titanic Survival Prediction API is running!"}

Port: 8001
Status: 401
Response: {"error":{"code":"MISSING_API_KEY","message":"API key is required"}}


In [37]:
import subprocess
import sys
import time
import requests

process_8002 = subprocess.Popen([
    sys.executable,
    "-m", "uvicorn",
    "main:app",
    "--app-dir", r"C:\Users\SASWAT",
    "--host", "127.0.0.1",
    "--port", "8002"
])

time.sleep(3)

print("Server running:", process_8002.poll() is None)

Server running: True


In [38]:
response = requests.get("http://127.0.0.1:8002/")

print("Status Code:", response.status_code)
print("Response:", response.json())

Status Code: 200
Response: {'message': 'Ride Booking Classification API is running!'}


In [40]:
##predicttion test

In [39]:
url = "http://127.0.0.1:8002/predict"

ride_data = {
    "vehicle_type": "Auto",
    "pickup_location": "AIIMS",
    "drop_location": "Anand Vihar",
    "booking_hour": 10,
    "day_of_week": 2,
    "is_weekend": 0,
    "month_name": "January"
}

response = requests.post(url, json=ride_data)

print("Status Code:", response.status_code)
print("Prediction:", response.json())

Status Code: 200
Prediction: {'predicted_booking_status': 'No Driver Found', 'prediction_probability': 0.214}


In [4]:
%whos

Variable               Type                  Data/Info
------------------------------------------------------
BaseModel              ModelMetaclass        <class 'pydantic.main.BaseModel'>
FastAPI                type                  <class 'fastapi.applications.FastAPI'>
Field                  function              <function Field at 0x000001518F5B07D0>
RideBookingInput       ModelMetaclass        <class '__main__.RideBookingInput'>
app                    FastAPI               <fastapi.applications.Fas<...>ct at 0x00000151925E86E0>
dataframe_columns      function              <function dataframe_colum<...>ns at 0x000001518F0038A0>
dataframe_hash         function              <function dataframe_hash at 0x000001518F0033D0>
dtypes_str             function              <function dtypes_str at 0x000001518F003ED0>
get_dataframes         function              <function get_dataframes at 0x000001518F003E20>
getpass                module                <module 'getpass' from 'C<...>conda3\\Li

In [7]:
import joblib

joblib.dump(model, "catboost_ride_booking_model.joblib")

print("Model saved successfully!")

Model saved successfully!


In [8]:
import os

print(os.path.exists("catboost_ride_booking_model.joblib"))

True


Start the srever

In [9]:
import subprocess
import sys

process = subprocess.Popen([
    sys.executable, "-m", "uvicorn",
    "main:app",
    "--host", "127.0.0.1",
    "--port", "8000"
])

print("FastAPI server started!")

FastAPI server started!


               prediction test

In [10]:
import joblib
import pandas as pd
import os

model_path = "catboost_ride_booking_model.joblib"

print("Model file exists:", os.path.exists(model_path))

model = joblib.load(model_path)

print("Model loaded successfully!")

Model file exists: True
Model loaded successfully!


In [13]:
##for new ride

In [14]:
new_ride = pd.DataFrame([{
    "vehicle_type": "Auto",
    "pickup_location": "AIIMS",
    "drop_location": "Anand Vihar",
    "booking_hour": 10,
    "day_of_week": 2,
    "is_weekend": 0,
    "month_name": "January"
}])

new_ride

,vehicle_type,pickup_location,drop_location,booking_hour,day_of_week,is_weekend,month_name
0,Auto,AIIMS,Anand Vihar,10,2,0,January


In [18]:
prediction = model.predict(new_ride)

print("Predicted Booking Status:", prediction[0][0])

Predicted Booking Status: No Driver Found


In [16]:
probabilities = model.predict_proba(new_ride)

classes = model.classes_

probability_df = pd.DataFrame({
    "Booking_Status": classes,
    "Probability": probabilities[0]
})

probability_df = probability_df.sort_values(
    by="Probability",
    ascending=False
)

probability_df

,Booking_Status,Probability
4,No Driver Found,0.213984
0,Cancelled by Customer,0.206048
3,Incomplete,0.200955
2,Completed,0.191687
1,Cancelled by Driver,0.187326


In [17]:
# Create multiple sample ride bookings

new_rides = pd.DataFrame([
    {
        "vehicle_type": "Auto",
        "pickup_location": "AIIMS",
        "drop_location": "Anand Vihar",
        "booking_hour": 10,
        "day_of_week": 2,
        "is_weekend": 0,
        "month_name": "January"
    },
    {
        "vehicle_type": "Bike",
        "pickup_location": "Airport",
        "drop_location": "Railway Station",
        "booking_hour": 18,
        "day_of_week": 5,
        "is_weekend": 1,
        "month_name": "February"
    },
    {
        "vehicle_type": "Car",
        "pickup_location": "City Center",
        "drop_location": "Airport",
        "booking_hour": 14,
        "day_of_week": 1,
        "is_weekend": 0,
        "month_name": "March"
    }
])

new_rides

,vehicle_type,pickup_location,drop_location,booking_hour,day_of_week,is_weekend,month_name
0,Auto,AIIMS,Anand Vihar,10,2,0,January
1,Bike,Airport,Railway Station,18,5,1,February
2,Car,City Center,Airport,14,1,0,March


In [19]:
# Predict booking status for multiple rides
predictions = model.predict(new_rides).reshape(-1)

# Calculate class probabilities
probabilities = model.predict_proba(new_rides)

# Create results table
results = new_rides.copy()

results["Predicted_Booking_Status"] = predictions
results["Prediction_Probability"] = probabilities.max(axis=1) * 100

results

,vehicle_type,pickup_location,drop_location,booking_hour,day_of_week,is_weekend,month_name,Predicted_Booking_Status,Prediction_Probability
0,Auto,AIIMS,Anand Vihar,10,2,0,January,No Driver Found,21.398372
1,Bike,Airport,Railway Station,18,5,1,February,Cancelled by Customer,28.364712
2,Car,City Center,Airport,14,1,0,March,Cancelled by Customer,37.132267
